In [1]:
import polars as pl

from libs.DataLoader import Loader
from libs.WeightedAvgModel import WeightedAvgModel
from libs.Solver import Solver
from libs.constants import *
from libs.utils import NDCG, count_polars

In [22]:
loader = Loader('ur0.01_ir0.01', content_embedding_size=64, batch_size=500_000)
(train_df, val_df), users_df, items_df = loader.load_data(convert_to_pandas=False)

Load metadata
Create lazy interaction datasets
Get unique users/items


  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Filter embeddings
Process metadata
Compute aggregates


  0%|          | 0/8 [00:00<?, ?it/s]

Filter interactions
Finalize interactions
Count
Train: 62_578 Val: 12_099 Users: 2_309 Items: 120_415


In [23]:
train_df: pl.LazyFrame = train_df.filter(pl.col(TARGET) > 0)  # Interaction: positive
val_df: pl.LazyFrame = val_df.filter(pl.col(TARGET) > 0)  # Interaction: positive

print(f"Train: {count_polars(train_df):_} Val: {count_polars(val_df):_}")

Train: 62_457 Val: 1_970


In [24]:
num_recent_videos = 100

agg = train_df \
    .sort(by=[pl.col(USER), pl.col(TIME_INDEX)], descending=[False, True]) \
    .group_by(USER) \
    .agg([
        pl.col(ITEM).slice(0, num_recent_videos).alias(ITEM),
        pl.col(TIME_INDEX).slice(0, num_recent_videos).alias(TIME_INDEX),
        pl.col(TARGET).slice(0, num_recent_videos).alias(TARGET)
    ])

val = val_df.select(ITEM).unique(ITEM)
cold_val = val.join(train_df.select(ITEM).unique(), on=ITEM, how="anti")

In [25]:
solver = Solver(500, 100, 101)
model = WeightedAvgModel('add', 'linear', alpha=0.5)
solver.collect_candidates(agg, val, items_df, model, 500, 500)

data:   0%|          | 0/5 [00:00<?, ?it/s]

val:   0%|          | 0/3 [00:00<?, ?it/s]

In [26]:
results = solver.solve()

In [27]:
predict = pl.DataFrame({
    ITEM: list(results.keys()),
    USER: [[u for u, _ in users_scores] for users_scores in results.values()]
})

In [28]:
for pred, pred_name in [
    (predict, "constrained")
]:
    for true, true_name in [
        (val_df, "all"),
        (val_df.join(cold_val, on=ITEM, how='inner'), "cold")
    ]:
        pl.LazyFrame().group_by().agg()
        print(f"Predict: {pred_name} Test: {true_name} NDCG: {NDCG(pred.to_pandas(), true.group_by(ITEM).agg(pl.col(USER)).collect().to_pandas()):.4f}")

Predict: constrained Test: all NDCG: 0.0942
Predict: constrained Test: cold NDCG: 0.1422
